In [1]:
import pickle
import json
import pandas as pd
import itertools
import os

with open('/ceph/hpc/home/euerikl/projects/fastighet/data/pipeline_test_data/pickle/results_rec.pkl', 'rb') as f:
    result_dict_rec = pickle.load(f)

with open('/ceph/hpc/home/euerikl/projects/fastighet/data/pipeline_test_data/pickle/results_det.pkl', 'rb') as f:
    result_dict_det = pickle.load(f)

rec_df = pd.read_csv('/ceph/hpc/home/euerikl/projects/fastighet/data/pipeline_test_data/recog/rec_file.txt', delimiter='|', usecols=[0], names=['file_name'])
rec_list = rec_df['file_name'].tolist()
#rec_list = [x.split('/')[1] for x in rec_list]


print(len(result_dict_det))
print(len(result_dict_rec))

2966
3232


In [2]:
flat_res_det = list()

for inst in result_dict_det:
    for annot in inst[0]:
        flat_res_det.append(annot[4])

In [ ]:


def write_jsons(results_det):

    with open('/ceph/hpc/home/euerikl/projects/fastighet/data/pipeline_test_data/coco/coco_for_det.json', 'r') as f:
        coco_dict = json.load(f)

    tuples = list(zip(coco_dict['images'], results_det))
    batches = dict()

    for key, group in itertools.groupby(tuples, key=lambda x: x[0]['file_name'].split('/')[0]):
        batches[key] = list(group)

    for key_batch, batch in batches.items():
               
        for inst in batch:
            page_name = inst[0]['file_name'].split('/')[1].split('.')[0]
            
            if len(inst[1][0]) == 0:
                
            
            for annot in inst[1][0]:
                entry = dict()
                entry['det_prob'] = str(annot[4])
                batch_dict[page_name].append(entry)

        s = json.dumps(batch_dict, indent = 4, ensure_ascii=False, sort_keys=True)
        f.write(str(s))

In [3]:
comb_tuple_list = list()

for name, res, det_prob in zip(rec_list, result_dict_rec, flat_res_det):
    sub = 0.0
    
    for sc in res['score']:
        sub += 0.27 - (10 * sc)
    
    score = 1.0-sub
    
    comb_tuple_list.append({'name': name, 'pred': res['text'], 'pred_conf': str(score), 'det_prob': str(det_prob)})


In [4]:
#first split on batches
batches = dict()

for key, group in itertools.groupby(comb_tuple_list, key=lambda x: x['name'].split('/')[0]):
        batches[key] = list(group)

for key_batch, batch in batches.items():

    json_out = os.path.join('/ceph/hpc/home/euerikl/projects/fastighet/data/pipeline_test_data/json', key_batch + '.json')
    with open(json_out, 'w', encoding='utf16') as f:

        json_dict = dict()

        for key_page, group in itertools.groupby(batch, key=lambda x: x['name']):
            page_dict = list(group)

            for json_inst in page_dict:
                json_inst.pop('name', None)

            json_dict[key_page.split('/')[1].split('.')[0]] = page_dict

        for i, inst in enumerate(result_dict_det):
            if len(inst[0]) == 0:
                page_str = key_batch + '_' + str(i+1).zfill(8)
                json_dict[page_str] = []

        s = json.dumps(json_dict, indent = 4, ensure_ascii=False, sort_keys=True)
        f.write(str(s))




In [20]:
for i, inst in enumerate(result_dict_det):
    if len(inst[0]) == 0:
        

2
3
4
6
8
10
12
14
20
22
32
40
42
46
52
54
56
64
66
68
72
76
80
84
88
132
172
180
188
250
252
284
430
436
440
450
452
456
460
464
466
470
474
476
478
480
575
576
578
579
580
1162
1199
1200
1202
1203
1204
1609
1610
1612
1613
1614
1854
2068
2281
2282
2284
2285
2286
2506
2568
2570
2576
2584
2592
2594
2612
2616
2620
2624
2634
2638
2642
2646
2652
2656
2658
2662
2666
2668
2670
2694
2698
2706
2708
2965
2966


In [ ]:
for res in result_dict:
    sub = 0.0
    for sc in res['score']:
        sub += 0.27-(10*sc)
    score = 1.0-sub
    print(score)
